In [8]:
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("../").resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
)

print("MLflow Tracking URI:", mlflow.get_tracking_uri())

model_name = "CustomerChurn_RF"
model_version = 1

model_uri = f"models:/{model_name}/{model_version}"

loaded_pipeline = mlflow.sklearn.load_model(model_uri)

print("Model loaded successfully.")
print(loaded_pipeline)

MLflow Tracking URI: sqlite:///C:\Users\cjc72\Desktop\e-commerce-ai-ml-retention-platform\e-commerce-ai-ml-retention-platform\mlflow.db
Model loaded successfully.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['frequency', 'monetary',
                                                   'avg_order_value',
                                                   'unique_categories',
                                                   'unique_sellers',
                                                   'avg_review_score',
                                                   'late_delivery_ratio',
                                                   'avg_installments',
                                                   'max_in

### Data Import

In [9]:
import pandas as pd 

PROJECT_ROOT = Path("../")

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "features" /
    "customer_churn_dataset.csv"
)


customers_ml = pd.read_csv(DATA_PATH)

customers_ml.head()

,frequency,monetary,avg_order_value,unique_categories,unique_sellers,avg_review_score,late_delivery_ratio,avg_installments,max_installments,payment_method_count,preferred_payment_type,state,latitude,longitude,churn_label
0,2,82.82,41.41,2,2,4.5,0.0,1.0,1.0,1,credit_card,SP,-23.577482,-46.587077,1
1,1,141.46,141.46,1,1,4.0,0.0,1.0,1.0,1,boleto,BA,-12.186877,-44.540232,0
2,1,179.12,179.12,1,1,5.0,0.0,3.0,3.0,1,credit_card,GO,-16.745150,-48.514783,0
3,1,72.20,72.20,1,1,5.0,0.0,1.0,1.0,1,credit_card,RN,-5.774002,-35.270976,1
4,1,28.62,28.62,1,1,5.0,0.0,1.0,1.0,1,credit_card,SP,-23.676257,-46.514580,1


In [10]:
customers_ml.shape

(96096, 15)

### Train Test Split

In [11]:
X = customers_ml.drop(columns=["churn_label"])

y = customers_ml["churn_label"]

In [12]:
X.value_counts()

frequency  monetary  avg_order_value  unique_categories  unique_sellers  avg_review_score  late_delivery_ratio  avg_installments  max_installments  payment_method_count  preferred_payment_type  state  latitude    longitude 
1          82.98     82.980000        1                  1               5.000000          0.000000             1.000000          1.0               1                     credit_card             SP     -23.604706  -46.693332    3
           42.27     42.270000        1                  1               5.000000          0.000000             1.000000          1.0               1                     credit_card             RN     -5.887088   -35.207494    3
           65.00     65.000000        1                  1               5.000000          0.000000             1.000000          1.0               1                     credit_card             PR     -25.434017  -49.297219    2
           0.00      0.000000         0                  0               1.000000        

In [13]:
y.value_counts()

churn_label
1    64212
0    31884
Name: count, dtype: int64

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (76876, 14)
Test set shape: (19220, 14)


### Test with 1 customer

In [16]:
sample_customer = X_test.iloc[[0]]

prediction = loaded_pipeline.predict(sample_customer)
probability = loaded_pipeline.predict_proba(sample_customer)[:, 1]

print("Prediction:", prediction[0])
print("Churn probability:", probability[0])

Prediction: 1
Churn probability: 0.5500257194790115


### Create Inference Module

In [17]:
# Verify the model signature
model_info = mlflow.models.get_model_info(model_uri)

print(model_info.signature)

inputs: 
  ['frequency': long (required), 'monetary': double (required), 'avg_order_value': double (required), 'unique_categories': long (required), 'unique_sellers': long (required), 'avg_review_score': double (required), 'late_delivery_ratio': double (required), 'avg_installments': double (required), 'max_installments': double (required), 'payment_method_count': long (required), 'preferred_payment_type': string (required), 'state': string (required), 'latitude': double (required), 'longitude': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



In [18]:
print(type(loaded_pipeline))
print(hasattr(loaded_pipeline, "predict_proba"))

<class 'sklearn.pipeline.Pipeline'>
True


In [19]:
test_probability = loaded_pipeline.predict_proba(X_test)

print(test_probability.shape)
print(test_probability[:5])

(19220, 2)
[[0.44997428 0.55002572]
 [0.61776542 0.38223458]
 [0.76635776 0.23364224]
 [0.61407899 0.38592101]
 [0.38791974 0.61208026]]


In [20]:
test_probability = loaded_pipeline.predict_proba(X_test)[:, 1]

print(test_probability[:10])
print(test_probability.min())
print(test_probability.max())

[0.55002572 0.38223458 0.23364224 0.38592101 0.61208026 0.21015791
 0.64365959 0.62378247 0.4544354  0.38944844]
0.009605948583580161
0.9969872340425532


### Deploy to FastAPI service
1. Check api/main.py

2. Run uvicorn api.main:app --reload

2. Test each of them

3. Run '''python -m pytest''' to run api/test_api.py

In [21]:
import mlflow
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
)


print("Tracking URI:")
print(mlflow.get_tracking_uri())

experiment = mlflow.get_experiment_by_name(
    "E-Commerce_Cust_Churn_Pred_Final_Model"
)

print("\nArtifact Location:")
print(experiment.artifact_location)

Tracking URI:
sqlite:///C:\Users\cjc72\Desktop\e-commerce-ai-ml-retention-platform\e-commerce-ai-ml-retention-platform\mlflow.db

Artifact Location:
file:///c:/Users/cjc72/Desktop/e-commerce-ai-ml-retention-platform/e-commerce-ai-ml-retention-platform/notebooks/mlruns/8


In [22]:
import mlflow
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
)


client = mlflow.tracking.MlflowClient()

model_name = "CustomerChurn_RF"

versions = client.search_model_versions(
    f"name='{model_name}'"
)

for version in versions:
    print("Version:", version.version)
    print("Run ID:", version.run_id)
    print("Source:", version.source)

Version: 1
Run ID: 3b37a11ebe8a4795a2036e4a035f6426
Source: models:/m-51144e404bb54e00a5a070785e0e22f5


In [23]:
import mlflow

model_uri = "models:/CustomerChurn_RF/1"

model_info = mlflow.models.get_model_info(model_uri)

print("Model URI:")
print(model_uri)

print("\nModel ID:")
print(model_info.model_id)

print("\nRun ID:")
print(model_info.run_id)

print("\nArtifact Path:")
print(model_info.artifact_path)

print("\nModel Flavor:")
print(model_info.flavors)

Model URI:
models:/CustomerChurn_RF/1

Model ID:
m-51144e404bb54e00a5a070785e0e22f5

Run ID:
3b37a11ebe8a4795a2036e4a035f6426

Artifact Path:
file:///c:/Users/cjc72/Desktop/e-commerce-ai-ml-retention-platform/e-commerce-ai-ml-retention-platform/notebooks/mlruns/8/models/m-51144e404bb54e00a5a070785e0e22f5/artifacts

Model Flavor:
{'python_function': {'env': {'conda': 'conda.yaml', 'virtualenv': 'python_env.yaml'}, 'loader_module': 'mlflow.sklearn', 'model_path': 'model.pkl', 'predict_fn': 'predict', 'python_version': '3.12.1'}, 'sklearn': {'code': None, 'pickled_model': 'model.pkl', 'serialization_format': 'pickle', 'sklearn_version': '1.9.0', 'skops_trusted_types': None}}


In [24]:
client = mlflow.tracking.MlflowClient()

model_version = client.get_model_version(
    name="CustomerChurn_RF",
    version="1"
)

print("Model version:")
print(model_version)

print("\nSource:")
print(model_version.source)

print("\nRun ID:")
print(model_version.run_id)

print("\nModel ID:")
print(model_version.model_id)

Model version:
<ModelVersion: aliases=[], creation_timestamp=1786532145476, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1786532145476, metrics=None, model_id=None, name='CustomerChurn_RF', params=None, run_id='3b37a11ebe8a4795a2036e4a035f6426', run_link=None, source='models:/m-51144e404bb54e00a5a070785e0e22f5', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

Source:
models:/m-51144e404bb54e00a5a070785e0e22f5

Run ID:
3b37a11ebe8a4795a2036e4a035f6426

Model ID:
None


In [25]:
loaded_model = mlflow.sklearn.load_model(
    "models:/CustomerChurn_RF/1"
)

print(type(loaded_model))

<class 'sklearn.pipeline.Pipeline'>


In [26]:
prediction = loaded_model.predict(X_test.head(1))

print(prediction)

[1]


# Local MLOps Deployment Pipeline

After registering the final `CustomerChurn_RF` model in MLflow Model Registry, the deployment workflow was automated locally to ensure that the registered model can be exported, validated, containerized, and tested consistently.

## 1. Export the Registered Model

The registered `CustomerChurn_RF` model is automatically loaded from MLflow Model Registry and exported as the deployment artifact:

```
MLflow Model Registry
↓
CustomerChurn_RF v1
↓
models/churn_model.pkl
```

`export_model.py` connects to the local MLflow tracking database, loads the approved model version, and saves the complete `sklearn.pipeline.Pipeline`.

```bash
python scripts/export_model.py
```

This ensures the deployed model is generated from the registered MLflow model rather than manually copying model files.

## 2. Verify the Exported Model

Before building the application image, the exported model is automatically validated.

The verification checks that:

- The model file exists.
- The model can be loaded successfully.
- The object provides `predict()`.
- The object provides `predict_proba()`.

```bash
python scripts/verify_model.py
```

Expected result:

Loaded model: <class 'sklearn.pipeline.Pipeline'>
Model verification passed.


This helps catch corrupted or incompatible model artifacts before deployment.

## 3. Build the Docker Image

The model verification and Docker build process were combined into a PowerShell automation script.

```powershell
.\scripts\build_image.ps1
```

The script performs:

```
Export model
↓
Verify model
↓
Build Docker image
```

The resulting image is:

ecommerce-churn-api:latest


The Docker image contains the FastAPI application, required dependencies, and the exported model artifact.

## 4. Containerized FastAPI Application

The Docker container packages the ML model and API into a reproducible deployment environment.

The application loads:

/app/models/churn_model.pkl


and exposes the FastAPI service on port 8000.

The container can be started with:

```bash
docker run --name ecommerce-churn-api -p 8000:8000 ecommerce-churn-api
```

This separates the deployed API from the local development environment and its MLflow database.

## 5. Automated API Testing

After starting the container, automated tests validate that the deployed API is functioning correctly.

The tests verify:

**Health endpoint**

GET /health

Confirms that the API and model are operational.

**Prediction endpoint**

POST /predict

Confirms that the deployed model can:

- Accept customer features.
- Generate a churn prediction.
- Return a valid churn probability.

Tests are executed with:

```bash
pytest tests/
```

Expected result:

2 passed


## 6. Local Deployment Pipeline

The complete automated local workflow is therefore:
```
            MLflow Model Registry
                     │
                     ▼
            CustomerChurn_RF v1
                     │
                     ▼
            export_model.py
                     │
                     ▼
            churn_model.pkl
                     │
                     ▼
            verify_model.py
                     │
                     ▼
             Docker Build
                     │
                     ▼
          ecommerce-churn-api
                     │
                     ▼
              FastAPI API
                     │
                     ▼
              Automated Tests
                     │
              ┌──────┴──────┐
              │             │
             PASS          FAIL
              │             │
              ▼             ▼
         Deployment       Stop
```

This provides a reproducible local workflow from registered ML model → deployment artifact → container → API validation, reducing the need for manual model handling and deployment steps.